In [1]:
import requests

def _prebuilt_placeholder(tool_data):
    def _run(*args, **kwargs):
        return {
            "tool_type": "prebuilt",
            "tool_id": tool_data["id"],
            "message": "Prebuilt tool execution placeholder"
        }
    return _run

def _custom_function_placeholder(tool_data):
    def _run(*args, **kwargs):
        pass
    return _run

In [2]:
from langchain_core.tools import StructuredTool
from pydantic import create_model
import requests
from typing import Dict, Any, List

def _custom_api_placeholder(tool_def: Dict[str, Any]) -> List:
    """
    Convert custom_api tool definition to LangChain tool
    
    Returns:  List containing one LangChain StructuredTool
    """
    name = tool_def["name"]
    description = tool_def["description"]
    api_url = tool_def["api_url"]
    api_request_type = tool_def["api_request_type"]
    custom_message = tool_def.get("custom_message", "")
    input_schema = tool_def["input_schema"]
    
    # Build Pydantic model from input_schema
    fields = {}
    properties = input_schema.get("properties", {})
    required = input_schema.get("required", [])
    
    for field_name, field_spec in properties.items():
        field_type_map = {
            "string": str,
            "number": float,
            "integer":  int,
            "boolean": bool
        }
        field_type = field_type_map.get(field_spec.get("type"), str)
        
        if field_name in required:
            fields[field_name] = (field_type, ...)
        else:
            fields[field_name] = (field_type, None)
    
    # Fallback if no properties
    if not fields: 
        fields = {"_placeholder": (str, None)}
    
    InputModel = create_model(f"{name}_Input", **fields)
    
    # API call function
    def execute_api_call(**kwargs) -> Dict[str, Any]:
        try:
            # Remove placeholder if exists
            kwargs.pop("_placeholder", None)
            
            if api_request_type.upper() == "GET":
                resp = requests.get(api_url, params=kwargs, timeout=10)
            else:  # POST
                resp = requests.post(api_url, json=kwargs, timeout=10)
            
            return {
                "status_code": resp.status_code,
                "data": resp.json() if resp.content else {},
                "custom_message": custom_message
            }
        except Exception as e:
            return {
                "status_code": 500,
                "data": {"error":  str(e)},
                "custom_message": custom_message
            }
    
    # Create LangChain tool
    tool = StructuredTool.from_function(
        func=execute_api_call,
        name=name,
        description=description,
        args_schema=InputModel
    )
    
    return [tool]

In [7]:
from typing import List
from langchain_core.tools import StructuredTool
from manager import ToolRegistryManager

def build_langchain_tools(tool_ids: List[str]) -> List[StructuredTool]:
    """
    Given a list of tool IDs, return LangChain-compatible Tool objects.
    """
    manager = ToolRegistryManager()
    langchain_tools: List[StructuredTool] = []

    for tool_id in tool_ids:
        tool_data = manager.get_tool(tool_id)
        if not tool_data:
            print("tool is missing")
            continue

        tool_type = tool_data.get("type")
        name = tool_data.get("name")
        description = tool_data.get("description")

        # --- Create tools based on type ---
        if tool_type == "prebuilt":
            func = _prebuilt_placeholder(tool_data)
            tool = StructuredTool.from_function(
                func=func,
                name=name,
                description=description
            )
            langchain_tools.append(tool)

        elif tool_type == "custom_function":
            func = _custom_function_placeholder(tool_data)
            tool = StructuredTool.from_function(
                func=func,
                name=name,
                description=description
            )
            langchain_tools.append(tool)

        elif tool_type == "custom_api":
            # _custom_api_placeholder returns a list of tools
            tools = _custom_api_placeholder(tool_data)
            langchain_tools.extend(tools)

    return langchain_tools

In [8]:
tool_ids = [
    "agent_8b8af64a-5063-4eb3-a00c-86c40e74ce43",
    "agent_84f2d4f0-97ad-457c-9f8b-6a70c8eb80af",
    "agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3"
]

tools = build_langchain_tools(tool_ids)

In [10]:
response = tools[2].invoke({"user_id": "123"})
response

{'status_code': 200,
 'data': [{'id': 'agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3',
   'type': 'custom_api',
   'name': 'get_user_orders',
   'description': 'Fetch orders',
   'input_schema': {'type': 'object'},
   'output_schema': {'type': 'object'},
   'custom_message': 'take only the order id and provide it to the user',
   'api_url': 'http://127.0.0.1:8000/v1/custom-api',
   'api_request_type': 'GET',
   'metadata': {'created_at': '2026-01-11T12:59:24.836428+00:00'}}],
 'custom_message': 'take only the order id and provide it to the user'}

In [ ]:
tool_def = {
    "id": "agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3",
    "type": "custom_api",
    "name": "get_user_orders",
    "description":  "Fetch orders",
    "input_schema": {},
    "custom_message": "take only the order id and provide it to the user",
    "api_url": "http://127.0.0.1:8000/v1/{prebuilt_id}",
    "api_request_type": "GET"
}

tool_list = _custom_api_placeholder(tool_def)
langchain_tool = tool_list[0]
result = langchain_tool.invoke({"user_id": "123"}) 

In [14]:
result

{'status_code': 200,
 'data': [{'id': 'agent_8b8af64a-5063-4eb3-a00c-86c40e74ce43',
   'type': 'prebuilt',
   'name': 'web_search',
   'description': 'Search the web',
   'input_schema': {'type': 'object'},
   'output_schema': {'type': 'object'},
   'metadata': {'created_at': '2026-01-11T06:29:45.719585+00:00'}},
  {'id': 'agent_84f2d4f0-97ad-457c-9f8b-6a70c8eb80af',
   'type': 'custom_function',
   'name': 'calculate_discount',
   'description': 'Calculate discount',
   'input_schema': {'type': 'object'},
   'output_schema': {'type': 'object'},
   'function': 'calculate_discount_fn',
   'metadata': {'created_at': '2026-01-11T06:29:45.719585+00:00'}},
  {'id': 'agent_f35d529c-5c1a-4bf5-9959-f501cbbdd9b3',
   'type': 'custom_api',
   'name': 'get_user_orders',
   'description': 'Fetch orders',
   'input_schema': {'type': 'object'},
   'output_schema': {'type': 'object'},
   'custom_message': 'take only the order id and provide it to the user',
   'api_url': 'http://127.0.0.1:8000/v1/cus

In [ ]:
# from langchain_openai import ChatOpenAI
# from langchain.agents import create_agent
# from dotenv import load_dotenv
# load_dotenv()

# from langchain_core.tools import StructuredTool
# from pydantic import BaseModel, Field

# class UserQuery(BaseModel):
#     user_id: int = Field(..., description="ID of the user")

# def user_info(user_id: int) -> dict:
#     if user_id == 123:
#         return {"user_id": "123", "name": "John Doe", "email": "john@123.com"}
#     if user_id == 456:
#         return {"user_id": "456", "name": "Jane Smith", "email": "jane@123.com"}
#     return {"error": "User not found"}

# tool1 = StructuredTool.from_function(
#     func=user_info,
#     name="user_info",
#     description="Get user information using user id",
#     args_schema=UserQuery
# )

# llm = ChatOpenAI(temperature=0)

# agent = create_agent(model=llm, tools=[tools[2]])

# response=agent.invoke({"messages": [("user", "Get info for user 123")]})
# response['messages'][-1].content